## Cellule 2 — Configuration des 3 connexions & extraction du référentiel CRM

In [100]:
import os
import warnings
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import warnings


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

try:
    oracledb.init_oracle_client()
except Exception as e:
    print(f"Oracle Client init warning (Thin mode will be used if failing): {e}")
    
MAPPING_SOCIETES = {
    "SICDA": "SICDA",
    "PHILEA": "PHILEA",
    "CAS": "CAS",
    "CMGP": "CMGP"
}

FILIALES = list(set(MAPPING_SOCIETES.values()))

DB_CONFIG_2 = {
    "user": os.getenv("DB_USER_1", "cas"),
    "password": os.getenv("DB_PASSWORD_1", "tiger"),
    "dsn": os.getenv("DB_DSN_1", "10.1.201.214/erpv6"),
    "schemas": ["CAS"]
}

DB_CONFIG_1 = {
    "user": os.getenv("DB_USER_2", "cmgp"),
    "password": os.getenv("DB_PASSWORD_2", "tiger"),
    "dsn": os.getenv("DB_DSN_2", "193.100.100.214/erpv6"),
    "schemas": ["CMGP", "SICDA", "PHILEA"] #c le principal
}

DB_CONFIG_CRM = {
    "user": os.getenv("DB_USER_CRM", "crm"),
    "password": os.getenv("DB_PASSWORD_CRM", "crm"),
    "dsn": os.getenv("DB_DSN_CRM", "193.100.100.102/erpv6"),
    "central_schema": "CRM"
}



print("Filiales configurées :", FILIALES)
def get_prod_engine():
    engines = {}
    
    try:
        dsn_crm = f"oracle+oracledb://{DB_CONFIG_CRM['user']}:{DB_CONFIG_CRM['password']}@{DB_CONFIG_CRM['dsn']}"
        engines["CRM"] = create_engine(dsn_crm)
    except Exception as e:
        print(f"Erreur d'initialisation du moteur CRM : {e}")

    try:
        dsn_1 = f"oracle+oracledb://{DB_CONFIG_1['user']}:{DB_CONFIG_1['password']}@{DB_CONFIG_1['dsn']}"
        engine_1 = create_engine(dsn_1)
        for schema in DB_CONFIG_1["schemas"]:
            engines[schema] = engine_1
    except Exception as e:
        print(f"Erreur d'initialisation du moteur ERP 1 : {e}")

    try:
        dsn_2 = f"oracle+oracledb://{DB_CONFIG_2['user']}:{DB_CONFIG_2['password']}@{DB_CONFIG_2['dsn']}"
        engine_2 = create_engine(dsn_2)
        for schema in DB_CONFIG_2["schemas"]:
            engines[schema] = engine_2
    except Exception as e:
        print(f"Erreur d'initialisation du moteur ERP 2 : {e}")
        
    return engines

conn = get_prod_engine()
print("Connexions Oracle créées avec succès.")
def run_sql(query, params=None, target_schema="CRM"):
    """
    Exécute une requête SQL sur le moteur approprié (CRM ou Schéma ERP cible).
    """
    engine = conn.get(target_schema)
    if not engine:
        print(f"Erreur : Aucun moteur de connexion trouvé pour la cible '{target_schema}'.")
        return pd.DataFrame()

    try:
        return pd.read_sql_query(query, engine, params=params)
    except Exception as e:
        print(f"\nErreur SQL sur le schéma {target_schema} :\n{e}")
        print("\nRequête exécutée :")
        print(query)
        return pd.DataFrame()

Filiales configurées : ['CAS', 'CMGP', 'SICDA', 'PHILEA']
Connexions Oracle créées avec succès.


In [101]:
print("Extraction des données de la table CRM.TIERS_SOCIETE...")

query_crm = """
SELECT 
    t.ID_CODE_SOCIETE,
    t.ID_CODE_TIERS_ERP_1,
    t.ID_LIB_TIERS_ERP,
    
    t.GST_AGENT_REPRESENTANT,
    r_rep.name AS AGENT_REPRESENTANT,
    CASE 
        WHEN t.GST_AGENT_REPRESENTANT = r_rep.ID THEN 'ACTUEL'
        WHEN t.GST_AGENT_REPRESENTANT = r_rep.OLD_ID THEN 'ANCIEN'
        ELSE 'INCONNU'
    END AS TYPE_CODE_REPRESENTANT,

    t.GST_AGENT_GESTIONNAIRE,
    r_gest.name AS AGENT_GESTIONNAIRE,
    CASE 
        WHEN t.GST_AGENT_GESTIONNAIRE = r_gest.ID THEN 'ACTUEL'
        WHEN t.GST_AGENT_GESTIONNAIRE = r_gest.OLD_ID THEN 'ANCIEN'
        ELSE 'INCONNU'
    END AS TYPE_CODE_GESTIONNAIRE

FROM crm.TIERS_SOCIETE t

LEFT JOIN crm.representant r_rep 
    ON t.GST_AGENT_REPRESENTANT IN (r_rep.ID, r_rep.OLD_ID)

LEFT JOIN crm.representant r_gest 
    ON t.GST_AGENT_GESTIONNAIRE IN (r_gest.ID, r_gest.OLD_ID)
"""

df_crm = run_sql(query_crm, target_schema="CRM")
df_crm.columns = df_crm.columns.str.upper()
if not df_crm.empty:
    print(f"Extraction CRM réussie : {len(df_crm)} lignes récupérées.")
else:
    print("Attention : La table CRM est vide ou une erreur est survenue.")

display(df_crm.head())

Extraction des données de la table CRM.TIERS_SOCIETE...
Extraction CRM réussie : 15004 lignes récupérées.


,ID_CODE_SOCIETE,ID_CODE_TIERS_ERP_1,ID_LIB_TIERS_ERP,GST_AGENT_REPRESENTANT,AGENT_REPRESENTANT,TYPE_CODE_REPRESENTANT,GST_AGENT_GESTIONNAIRE,AGENT_GESTIONNAIRE,TYPE_CODE_GESTIONNAIRE
0,CAS,DD00MAT1003212,ATLANTIC AGRO AU,RA01017,WAHROUR NOUREDDINE,ACTUEL,NaN,NaN,INCONNU
1,SICDA,IE00MAU1000399,SOS IRRIGATION ET POMPAGE,RS15055,BEROUAL MOHAMED,ACTUEL,NaN,NaN,INCONNU
2,SICDA,IE00MAU1000040,SAMADY S.N.C,NaN,NaN,INCONNU,NaN,NaN,INCONNU
3,SICDA,IE00MAU1003693,WATEGA,RS18068,BOUSSEDRA AZIZ,ACTUEL,NaN,NaN,INCONNU
4,SICDA3,IE00MAU2001582,OUARDIGHA,NaN,NaN,INCONNU,NaN,NaN,INCONNU


In [102]:
import numpy as np
import pandas as pd

anomalie_representant_dict = {}


print("Début du traitement et des comparaisons par société...\n")

for societe_crm, schema_erp in MAPPING_SOCIETES.items():
    print(f"--- Traitement de la société : {societe_crm} (Schéma ERP : {schema_erp}) ---")
    
    df_crm_soc = df_crm[df_crm['ID_CODE_SOCIETE'] == societe_crm]
    if df_crm_soc.empty:
        continue

    query_erp = f"""
    SELECT 
    b.BPCNUM_0, 
    b.REP_0, 
    s1.REPNAM_0 AS name_rep_0, 
    b.REP_1,  
    s2.REPNAM_0 AS name_rep_1
    FROM {schema_erp}.BPCUSTOMER b 
    
    LEFT JOIN {schema_erp}.salesrep s1 
        ON b.REP_0 = s1.REPNUM_0
    
    LEFT JOIN {schema_erp}.salesrep s2 
        ON b.REP_1 = s2.REPNUM_0
        
    WHERE b.BPCSTA_0 = 2
    """
    df_erp_soc = run_sql(query_erp, target_schema=schema_erp)
    df_erp_soc.columns = df_erp_soc.columns.str.upper()

    if df_erp_soc.empty:
        print(f"  -> Table ERP vide ou inaccessible pour le schéma {schema_erp}. Passage.\n")
        continue

    df_merged = pd.merge(
        df_crm_soc, 
        df_erp_soc, 
        left_on='ID_CODE_TIERS_ERP_1', 
        right_on='BPCNUM_0', 
        how='inner'
    )

    if df_merged.empty:
        print(f"  -> Aucun tiers en commun trouvé après jointure pour {societe_crm}.\n")
        continue

    print(f"  -> {len(df_merged)} tiers communs trouvés.")

    def clean_spaces_to_nan(series):
        if series.dtype == 'object' or pd.api.types.is_string_dtype(series.dtype):
            s_str = series.astype(str).str.strip()
            s_clean = s_str.replace(['nan', 'None', ''], np.nan)
            s_clean[series.isna()] = np.nan
            return s_clean
        return series

    # ==========================================
    # TRAITEMENT : REPRÉSENTANT (GST_AGENT_REPRESENTANT vs REP_0)
    # ==========================================
    col_crm_rep = clean_spaces_to_nan(df_merged['GST_AGENT_REPRESENTANT'])
    col_erp_rep = clean_spaces_to_nan(df_merged['REP_0'])


    anomalie1_rep_mask = (col_crm_rep.notna() & col_erp_rep.notna() & (col_crm_rep != col_erp_rep))
    df_anom1_rep = df_merged[anomalie1_rep_mask].copy()
    if not df_anom1_rep.empty:
        df_anom1_rep = df_anom1_rep[[
            'ID_CODE_SOCIETE', 'ID_CODE_TIERS_ERP_1', 'ID_LIB_TIERS_ERP', 
            'GST_AGENT_REPRESENTANT', 'AGENT_REPRESENTANT','TYPE_CODE_REPRESENTANT' ,'REP_0','NAME_REP_0'
        ]]
        df_anom1_rep.columns = [
            'ID_CODE_SOCIETE', 'ID_CODE_TIERS_ERP_1', 'LIB', 
            'CODE_REPRESENTANT_CRM','NOM_REPRESENTANT_CRM','TYPE_CODE_REPRESENTANT_CRM' ,'CODE_REPRESENTANT_ERP','NOM_REPRESENTANT_ERP'
        ]
        anomalie_representant_dict[societe_crm] = df_anom1_rep
        print(f"  -> Représentant : {len(df_anom1_rep)} anomalie(s) de type 1 (non vides et différents).")




Début du traitement et des comparaisons par société...

--- Traitement de la société : SICDA (Schéma ERP : SICDA) ---
  -> 945 tiers communs trouvés.
  -> Représentant : 6 anomalie(s) de type 1 (non vides et différents).
--- Traitement de la société : PHILEA (Schéma ERP : PHILEA) ---
  -> 708 tiers communs trouvés.
  -> Représentant : 42 anomalie(s) de type 1 (non vides et différents).
--- Traitement de la société : CAS (Schéma ERP : CAS) ---
  -> 5626 tiers communs trouvés.
  -> Représentant : 157 anomalie(s) de type 1 (non vides et différents).
--- Traitement de la société : CMGP (Schéma ERP : CMGP) ---
  -> 6821 tiers communs trouvés.
  -> Représentant : 402 anomalie(s) de type 1 (non vides et différents).


In [103]:
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

REGEX_CODE = r"[A-Z]{2}[0-9]{5}"     

ERP_GROUPE_PAR_SOCIETE = {"SICDA": "ERP_PRINCIPAL", "PHILEA": "ERP_PRINCIPAL",
                          "CMGP": "ERP_PRINCIPAL", "CAS": "ERP_CAS"}
SOCIETES_ERP = set(ERP_GROUPE_PAR_SOCIETE)   
DOSSIER_RUNS = Path("runs")

CRM_TABLE, CRM_COL_REP, CRM_COL_CLIENT = "tiers_societe", "GST_AGENT_REPRESENTANT", "ID_CODE_TIERS_ERP_1"
CRM_COL_SOCIETE = "ID_CODE_SOCIETE"   

ERP_TABLE, ERP_COL_CLIENT = "bpcustomer", "bpcnum_0"
ERP_COL_REP_DEFAUT = "rep_0"

ERP_COL_REP_PAR_SOCIETE = {"CMGP": "XREPR1_0"}
def col_rep_erp(societe: str) -> str:
    return ERP_COL_REP_PAR_SOCIETE.get(societe, ERP_COL_REP_DEFAUT)

COLS_ENTREE = ["ID_CODE_SOCIETE", "ID_CODE_TIERS_ERP_1", "LIB",
               "CODE_REPRESENTANT_CRM", "NOM_REPRESENTANT_CRM", "TYPE_CODE_REPRESENTANT_CRM",
               "CODE_REPRESENTANT_ERP", "NOM_REPRESENTANT_ERP"]
SRC_CRM, SRC_ERP, SRC_ANOMALIE, SRC_NON_TRAITE = "CRM", "ERP", "ANOMALIE_FORMAT", "NON_TRAITE"

In [104]:
def preparer_entree(d: dict) -> pd.DataFrame:
    frames = []
    for cle, df in d.items():
        soc = str(cle).strip().upper()
        if soc not in SOCIETES_ERP:
            raise ValueError(f"Société inconnue dans le dict : {cle!r} (attendu : {sorted(SOCIETES_ERP)})")
        manq = set(COLS_ENTREE) - set(df.columns) - {"ID_CODE_SOCIETE"}
        if manq:
            raise ValueError(f"[{soc}] colonnes manquantes : {sorted(manq)}")
        f = df.copy()
        f["ID_CODE_SOCIETE"] = soc          
        frames.append(f[COLS_ENTREE])
    return pd.concat(frames, ignore_index=True)

df_entree = preparer_entree(anomalie_representant_dict)
print(df_entree.groupby("ID_CODE_SOCIETE").size())
df_entree

ID_CODE_SOCIETE
CAS       157
CMGP      402
PHILEA     42
SICDA       6
dtype: int64


,ID_CODE_SOCIETE,ID_CODE_TIERS_ERP_1,LIB,CODE_REPRESENTANT_CRM,NOM_REPRESENTANT_CRM,TYPE_CODE_REPRESENTANT_CRM,CODE_REPRESENTANT_ERP,NOM_REPRESENTANT_ERP
0,SICDA,IE00MAU1006872,CTRAVAUX,RP009,NaN,INCONNU,RS18068,BOUSSEDRA AZIZ
1,SICDA,IE00MAU2000527,LES DIVERS AOURACHS MAROCAINS,RS15055,BEROUAL MOHAMED,ACTUEL,RS22104,HICHAM SLAMA
2,SICDA,IE00MAU2000669,ACHARIKA TRAVAUX,RS15055,BEROUAL MOHAMED,ACTUEL,RS18068,BOUSSEDRA AZIZ
3,SICDA,IE00MAU2003766,V.COMME VERT,RS15055,BEROUAL MOHAMED,ACTUEL,RS22104,HICHAM SLAMA
4,SICDA,IE00MAU2000040,SAMADY S.N.C,RS15055,BEROUAL MOHAMED,ACTUEL,RS18068,BOUSSEDRA AZIZ
...,...,...,...,...,...,...,...,...
602,CMGP,DR00MAA1010956,PHYTO FERME MAROC,RC24145,REDOUANE NIMOUSSA,ACTUEL,RC97012,ELHAMAHAM MOHAMED
603,CMGP,DR00MAS1000473,ML GREEN GROUP,RC08031,KHALID ZAZA,ACTUEL,RC25125,Brahim LARZA
604,CMGP,PP30MAP1002251,DOMAINES AGRICOLES SIDI KACEM,RC96010,KHATTAB HICHAM,ACTUEL,RC18067,FATIMA ZAHRA KAZBOURI
605,CMGP,DR00MAS1010671,ZEROUAL ABDELLAH,RPS009,NaN,INCONNU,RC25125,Brahim LARZA


In [105]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
DOSSIER = DOSSIER_RUNS / RUN_ID
DOSSIER.mkdir(parents=True, exist_ok=True)

df_entree.to_csv(DOSSIER / "backup_entree.csv", index=False, encoding="utf-8-sig")
print("Sauvegarde :", DOSSIER / "backup_entree.csv", "| RUN_ID =", RUN_ID)

Sauvegarde : runs/20260929_170933/backup_entree.csv | RUN_ID = 20260929_170933


In [107]:
def _est_vide(s):  return s.isna() | s.astype("string").str.strip().fillna("").eq("")
def _est_valide(s): return s.astype("string").str.fullmatch(REGEX_CODE).fillna(False).astype(bool)

def arbitrer(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    crm_code, erp_code = out["CODE_REPRESENTANT_CRM"], out["CODE_REPRESENTANT_ERP"]
    crm_vide, erp_vide = _est_vide(out["NOM_REPRESENTANT_CRM"]), _est_vide(out["NOM_REPRESENTANT_ERP"])
    crm_ok, erp_ok = _est_valide(crm_code), _est_valide(erp_code)

    deux_vides = crm_vide & erp_vide
    crm_seul, erp_seul = ~crm_vide & erp_vide, crm_vide & ~erp_vide
    deux = ~crm_vide & ~erp_vide

    regles = [
        ("R1_NOMS_VIDES",           deux_vides,                  SRC_NON_TRAITE),
        ("R2_CRM_SEUL_OK",          crm_seul & crm_ok,           SRC_CRM),
        ("R2_CRM_SEUL_KO",          crm_seul & ~crm_ok,          SRC_ANOMALIE),
        ("R2_ERP_SEUL_OK",          erp_seul & erp_ok,           SRC_ERP),
        ("R2_ERP_SEUL_KO",          erp_seul & ~erp_ok,          SRC_ANOMALIE),
        ("R3_DEUX_OK_PRIORITE_ERP", deux & crm_ok & erp_ok,      SRC_ERP),
        ("R3_SEUL_CRM_OK",          deux & crm_ok & ~erp_ok,     SRC_CRM),
        ("R3_SEUL_ERP_OK",          deux & ~crm_ok & erp_ok,     SRC_ERP),
        ("R3_AUCUN_OK",             deux & ~crm_ok & ~erp_ok,    SRC_ANOMALIE),
    ]
    conds = [r[1].to_numpy() for r in regles]
    out["REGLE_APPLIQUEE"] = np.select(conds, [r[0] for r in regles], default="INCONNUE")
    out["SOURCE_VERITE_ARBITREE"] = np.select(conds, [r[2] for r in regles], default=SRC_NON_TRAITE)

    src = out["SOURCE_VERITE_ARBITREE"]
    out["CODE_REPRESENTANT_ARBITRE"] = np.select(
        [src.eq(SRC_CRM).to_numpy(), src.eq(SRC_ERP).to_numpy()],
        [crm_code.to_numpy(dtype=object), erp_code.to_numpy(dtype=object)], default=None).astype(object)
    out.loc[src.isin([SRC_ANOMALIE, SRC_NON_TRAITE]), "CODE_REPRESENTANT_ARBITRE"] = None

    resolu = out["CODE_REPRESENTANT_ARBITRE"].notna()
    out["MAJ_CRM_NECESSAIRE"] = resolu & out["CODE_REPRESENTANT_ARBITRE"].ne(crm_code)
    out["MAJ_ERP_NECESSAIRE"] = resolu & out["CODE_REPRESENTANT_ARBITRE"].ne(erp_code)
    out["DATE_ARBITRAGE"] = datetime.now().isoformat(timespec="seconds")
    return out

df_arbitre = arbitrer(df_entree)

resultat_dict = {soc: g.reset_index(drop=True) for soc, g in df_arbitre.groupby("ID_CODE_SOCIETE")}

print(df_arbitre["SOURCE_VERITE_ARBITREE"].value_counts())
df_arbitre[["ID_CODE_SOCIETE", "ID_CODE_TIERS_ERP_1", "CODE_REPRESENTANT_ARBITRE",
            "SOURCE_VERITE_ARBITREE", "REGLE_APPLIQUEE"]]


SOURCE_VERITE_ARBITREE
ERP                498
CRM                101
ANOMALIE_FORMAT      8
Name: count, dtype: int64


,ID_CODE_SOCIETE,ID_CODE_TIERS_ERP_1,CODE_REPRESENTANT_ARBITRE,SOURCE_VERITE_ARBITREE,REGLE_APPLIQUEE
0,SICDA,IE00MAU1006872,RS18068,ERP,R2_ERP_SEUL_OK
1,SICDA,IE00MAU2000527,RS22104,ERP,R3_DEUX_OK_PRIORITE_ERP
2,SICDA,IE00MAU2000669,RS18068,ERP,R3_DEUX_OK_PRIORITE_ERP
3,SICDA,IE00MAU2003766,RS22104,ERP,R3_DEUX_OK_PRIORITE_ERP
4,SICDA,IE00MAU2000040,RS18068,ERP,R3_DEUX_OK_PRIORITE_ERP
...,...,...,...,...,...
602,CMGP,DR00MAA1010956,RC97012,ERP,R3_DEUX_OK_PRIORITE_ERP
603,CMGP,DR00MAS1000473,RC25125,ERP,R3_DEUX_OK_PRIORITE_ERP
604,CMGP,PP30MAP1002251,RC18067,ERP,R3_DEUX_OK_PRIORITE_ERP
605,CMGP,DR00MAS1010671,RC25125,ERP,R2_ERP_SEUL_OK


In [108]:
audit = pd.DataFrame({
    "RUN_ID": RUN_ID,
    "ID_CODE_SOCIETE": df_arbitre["ID_CODE_SOCIETE"],
    "ID_CODE_TIERS_ERP_1": df_arbitre["ID_CODE_TIERS_ERP_1"],
    "ANCIEN_CODE_CRM": df_arbitre["CODE_REPRESENTANT_CRM"],
    "ANCIEN_CODE_ERP": df_arbitre["CODE_REPRESENTANT_ERP"],
    "CODE_RETENU": df_arbitre["CODE_REPRESENTANT_ARBITRE"],
    "SOURCE_RETENUE": df_arbitre["SOURCE_VERITE_ARBITREE"],
    "REGLE_APPLIQUEE": df_arbitre["REGLE_APPLIQUEE"],
    "HORODATAGE": df_arbitre["DATE_ARBITRAGE"],
})
anomalies = df_arbitre[df_arbitre["SOURCE_VERITE_ARBITREE"].isin([SRC_ANOMALIE, SRC_NON_TRAITE])].copy()

audit.to_csv(DOSSIER / "audit_arbitrage.csv", index=False, encoding="utf-8-sig")
anomalies.to_csv(DOSSIER / "anomalies.csv", index=False, encoding="utf-8-sig")
print(f"Audit : {len(audit)} lignes | Anomalies : {len(anomalies)} lignes")
anomalies

Audit : 607 lignes | Anomalies : 8 lignes


,ID_CODE_SOCIETE,ID_CODE_TIERS_ERP_1,LIB,CODE_REPRESENTANT_CRM,NOM_REPRESENTANT_CRM,TYPE_CODE_REPRESENTANT_CRM,CODE_REPRESENTANT_ERP,NOM_REPRESENTANT_ERP,REGLE_APPLIQUEE,SOURCE_VERITE_ARBITREE,CODE_REPRESENTANT_ARBITRE,MAJ_CRM_NECESSAIRE,MAJ_ERP_NECESSAIRE,DATE_ARBITRAGE
32,PHILEA,DZ00MAP1008437,ECHANTILLONS COMMERCIAL,CPHC02,NaN,INCONNU,CPHC05,MERSSI FATIMA ZAHRA,R2_ERP_SEUL_KO,ANOMALIE_FORMAT,NaN,False,False,2026-09-29T17:10:08
66,CAS,DD00MAK1007741,TAKAMOUL AL FALLAH,LREP0005,NaN,INCONNU,KREP0006,ELOUARDI ABDELKARIM,R2_ERP_SEUL_KO,ANOMALIE_FORMAT,NaN,False,False,2026-09-29T17:10:08
76,CAS,DD00MAK1008063,DOMAINE DE BAHRAOUI,LREP0005,NaN,INCONNU,KREP0006,ELOUARDI ABDELKARIM,R2_ERP_SEUL_KO,ANOMALIE_FORMAT,NaN,False,False,2026-09-29T17:10:08
77,CAS,DD00MAK1008066,CIVILE AGRICOLE DOMAINE DE MAAMOURA,LREP0005,NaN,INCONNU,KREP0006,ELOUARDI ABDELKARIM,R2_ERP_SEUL_KO,ANOMALIE_FORMAT,NaN,False,False,2026-09-29T17:10:08
86,CAS,DA00MAL1008369,BERTANI MO,AREP0015,NaN,INCONNU,LREP0001,LARACHE,R2_ERP_SEUL_KO,ANOMALIE_FORMAT,NaN,False,False,2026-09-29T17:10:08
99,CAS,DA00MAA2008368,EL MOKRIFI,AREP0015,NaN,INCONNU,LREP0005,LGOU HAMID,R2_ERP_SEUL_KO,ANOMALIE_FORMAT,NaN,False,False,2026-09-29T17:10:08
439,CMGP,PP10MAJ1005591,MOHAMED SAID MASRAR,RPJ105,NaN,INCONNU,RPJ104,EL ARIM MAROUANE,R2_ERP_SEUL_KO,ANOMALIE_FORMAT,NaN,False,False,2026-09-29T17:10:08
472,CMGP,DR00MAR1002463,ORIENT VERT IRRIGATION,RPS009,NaN,INCONNU,RPR206,YASSINE HANNI,R2_ERP_SEUL_KO,ANOMALIE_FORMAT,NaN,False,False,2026-09-29T17:10:08


In [97]:
crm_engine = conn.get("CRM")
erp_engine = conn.get("CMGP")
erp_cas_engine = conn.get("CAS")

erp_engines = {"ERP_PRINCIPAL": erp_engine, "ERP_CAS": erp_cas_engine}

In [109]:
def _sql_crm():
    filtre = f" AND {CRM_COL_SOCIETE} = :societe" if CRM_COL_SOCIETE else ""
    return f"UPDATE {CRM_TABLE} SET {CRM_COL_REP} = :val WHERE {CRM_COL_CLIENT} = :client{filtre}"

ERP_TABLE_DLV = "bpdlvcust"   # adresses de livraison client (Sage X3)
# Colonne représentant dans BPDLVCUST, par société (défaut rep_0 pour TOUTES les sociétés, y compris CMGP :
# contrairement à BPCUSTOMER où CMGP utilise XREPR1_0). Surcharger ici si besoin : {"CMGP": "..."}
ERP_COL_REP_DLV_PAR_SOCIETE = {}
def col_rep_dlv(societe: str) -> str:
    return ERP_COL_REP_DLV_PAR_SOCIETE.get(societe, ERP_COL_REP_DEFAUT)

def _sql_erp_dlv(societe, aligne=False):
    """Version TOUT ÉCRASER : toutes les adresses de livraison du client prennent le code arbitré.
    aligne=True (non utilisé ici) restreindrait aux adresses dont le représentant == ancien code (:ancien)."""
    if societe not in SOCIETES_ERP:
        raise ValueError(f"Société ERP non autorisée : {societe!r}")
    col = col_rep_dlv(societe)
    sql = f"UPDATE {societe}.{ERP_TABLE_DLV} SET {col} = :val WHERE {ERP_COL_CLIENT} = :client"
    if aligne:
        sql += f" AND (({col} = :ancien) OR ({col} IS NULL AND :ancien IS NULL))"
    return sql

def _sql_erp(societe):
    if societe not in SOCIETES_ERP:
        raise ValueError(f"Société ERP non autorisée : {societe!r}")
    return f"UPDATE {societe}.{ERP_TABLE} SET {col_rep_erp(societe)} = :val WHERE {ERP_COL_CLIENT} = :client"

def appliquer_mises_a_jour(df, run_id, crm_engine, erp_engines, dry_run=True):
    log, horo, bk_dlv = [], datetime.now().isoformat(timespec="seconds"), []
    def _l(systeme, connexion, r, ancien, statut, rc, rc_dlv=None, nb_dlv=None):
        return {"RUN_ID": run_id, "SYSTEME": systeme, "CONNEXION": connexion,
                "ID_CODE_SOCIETE": r.ID_CODE_SOCIETE, "ID_CODE_TIERS_ERP_1": r.ID_CODE_TIERS_ERP_1,
                "ANCIENNE_VALEUR": ancien, "NOUVELLE_VALEUR": r.CODE_REPRESENTANT_ARBITRE,
                "STATUT": statut, "ROWCOUNT": rc, "ROWCOUNT_BPDLVCUST": rc_dlv, "NB_ADRESSES_BPDLVCUST": nb_dlv, "HORODATAGE": horo}
    try:
        l_crm = df[df["MAJ_CRM_NECESSAIRE"]]
        if len(l_crm):
            local = []
            if dry_run:
                local = [_l("CRM", "CRM", r, r.CODE_REPRESENTANT_CRM, "DRY_RUN", None) for r in l_crm.itertuples()]
            else:
                with crm_engine.begin() as conn:
                    for r in l_crm.itertuples():
                        p = {"val": r.CODE_REPRESENTANT_ARBITRE, "client": r.ID_CODE_TIERS_ERP_1}
                        if CRM_COL_SOCIETE: p["societe"] = r.ID_CODE_SOCIETE
                        rc = conn.execute(text(_sql_crm()), p).rowcount
                        local.append(_l("CRM", "CRM", r, r.CODE_REPRESENTANT_CRM, "APPLIQUE", rc))
            log.extend(local)   # ajouté au log seulement si la transaction a été validée

        l_erp = df[df["MAJ_ERP_NECESSAIRE"]]
        for groupe, sous in l_erp.groupby(l_erp["ID_CODE_SOCIETE"].map(ERP_GROUPE_PAR_SOCIETE)):
            local = []
            if dry_run:
                local = [_l("ERP", groupe, r, r.CODE_REPRESENTANT_ERP, "DRY_RUN", None) for r in sous.itertuples()]
            else:
                with erp_engines[groupe].begin() as conn:
                    for r in sous.itertuples():
                        p = {"val": r.CODE_REPRESENTANT_ARBITRE, "client": r.ID_CODE_TIERS_ERP_1}
                        # sauvegarde des valeurs actuelles de BPDLVCUST (nécessaire au rollback)
                        col = col_rep_dlv(r.ID_CODE_SOCIETE)
                        ancien = None if pd.isna(r.CODE_REPRESENTANT_ERP) else r.CODE_REPRESENTANT_ERP
                        sel = f"SELECT bpaadd_0, {col} FROM {r.ID_CODE_SOCIETE}.{ERP_TABLE_DLV} WHERE {ERP_COL_CLIENT} = :client"
                        adresses = conn.execute(text(sel), {"client": r.ID_CODE_TIERS_ERP_1}).fetchall()
                        for adr, val_orig in adresses:
                            bk_dlv.append({"ID_CODE_SOCIETE": r.ID_CODE_SOCIETE, "ID_CODE_TIERS_ERP_1": r.ID_CODE_TIERS_ERP_1,
                                           "BPAADD": adr, "VALEUR_ORIG": val_orig,
                                           "ALIGNE": True})   # toutes les adresses sont mises à jour (utilisé par le rollback)
                        rc = conn.execute(text(_sql_erp(r.ID_CODE_SOCIETE)), p).rowcount          # BPCUSTOMER
                        # BPDLVCUST : TOUTES les adresses de livraison du client prennent le code arbitré
                        p_dlv = {"val": r.CODE_REPRESENTANT_ARBITRE, "client": r.ID_CODE_TIERS_ERP_1}
                        rc_dlv = conn.execute(text(_sql_erp_dlv(r.ID_CODE_SOCIETE)), p_dlv).rowcount
                        local.append(_l("ERP", groupe, r, r.CODE_REPRESENTANT_ERP, "APPLIQUE", rc, rc_dlv, len(adresses)))
            log.extend(local)
    finally:
        if bk_dlv:
            pd.DataFrame(bk_dlv).to_csv(DOSSIER_RUNS / run_id / "backup_bpdlvcust.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame(log).to_csv(DOSSIER_RUNS / run_id / "update_log.csv", index=False, encoding="utf-8-sig")
    return pd.DataFrame(log)

log_maj = appliquer_mises_a_jour(df_arbitre, RUN_ID, crm_engine, erp_engines, dry_run=False)
log_maj

,RUN_ID,SYSTEME,CONNEXION,ID_CODE_SOCIETE,ID_CODE_TIERS_ERP_1,ANCIENNE_VALEUR,NOUVELLE_VALEUR,STATUT,ROWCOUNT,HORODATAGE
0,20260929_170933,CRM,CRM,SICDA,IE00MAU1006872,RP009,RS18068,APPLIQUE,1,2026-09-29T17:10:22
1,20260929_170933,CRM,CRM,SICDA,IE00MAU2000527,RS15055,RS22104,APPLIQUE,1,2026-09-29T17:10:22
2,20260929_170933,CRM,CRM,SICDA,IE00MAU2000669,RS15055,RS18068,APPLIQUE,1,2026-09-29T17:10:22
3,20260929_170933,CRM,CRM,SICDA,IE00MAU2003766,RS15055,RS22104,APPLIQUE,1,2026-09-29T17:10:22
4,20260929_170933,CRM,CRM,SICDA,IE00MAU2000040,RS15055,RS18068,APPLIQUE,1,2026-09-29T17:10:22
...,...,...,...,...,...,...,...,...,...,...
594,20260929_170933,ERP,ERP_PRINCIPAL,CMGP,DA00MAM1004214,RPM403,RC08032,APPLIQUE,1,2026-09-29T17:10:22
595,20260929_170933,ERP,ERP_PRINCIPAL,CMGP,DR00MAN1000615,RPN905,RC18063,APPLIQUE,1,2026-09-29T17:10:22
596,20260929_170933,ERP,ERP_PRINCIPAL,CMGP,DR00MAJ1003740,RPJ107,RC13053,APPLIQUE,1,2026-09-29T17:10:22
597,20260929_170933,ERP,ERP_PRINCIPAL,CMGP,DD00MAP1000765,RPP037,RC19069,APPLIQUE,1,2026-09-29T17:10:22


In [98]:
# 2) Application réelle (décommenter après validation du dry-run)
# log_maj = appliquer_mises_a_jour(df_arbitre, RUN_ID, crm_engine, erp_engines, dry_run=False)
# log_maj

## 1) Simulation
#log_maj = appliquer_mises_a_jour(df_arbitre, RUN_ID, crm_engine, erp_engines, dry_run=True)
#log_maj

In [2]:
def rollback(run_id, crm_engine, erp_engines, dry_run=True, forcer=False):
    d = DOSSIER_RUNS / run_id
    log = pd.read_csv(d / "update_log.csv", dtype=str, keep_default_na=False, na_values=[""])
    bk = (pd.read_csv(d / "backup_entree.csv", dtype=str, keep_default_na=False, na_values=[""])
            .drop_duplicates(["ID_CODE_SOCIETE", "ID_CODE_TIERS_ERP_1"])
            .set_index(["ID_CODE_SOCIETE", "ID_CODE_TIERS_ERP_1"]))
    f_dlv = d / "backup_bpdlvcust.csv"
    bk_dlv = pd.read_csv(f_dlv, dtype=str, keep_default_na=False, na_values=[""]) if f_dlv.exists() else None
    a_annuler = log[log["STATUT"] == "APPLIQUE"]
    res = []

    def garde(sql, col):
        return sql + f" AND (({col} = :attendu) OR ({col} IS NULL AND :attendu IS NULL))"

    for connexion, sous in a_annuler.groupby("CONNEXION"):
        systeme = sous["SYSTEME"].iloc[0]
        col_orig = "CODE_REPRESENTANT_CRM" if systeme == "CRM" else "CODE_REPRESENTANT_ERP"
        taches = []   # (r, sql, params, valeur_restauree, table, adresse)
        for r in sous.itertuples():
            orig = bk.loc[(r.ID_CODE_SOCIETE, r.ID_CODE_TIERS_ERP_1), col_orig]
            orig = None if pd.isna(orig) else orig
            col_rep = CRM_COL_REP if systeme == "CRM" else col_rep_erp(r.ID_CODE_SOCIETE)
            sql = _sql_crm() if systeme == "CRM" else _sql_erp(r.ID_CODE_SOCIETE)
            if not forcer: sql = garde(sql, col_rep)
            p = {"val": orig, "client": r.ID_CODE_TIERS_ERP_1, "attendu": r.NOUVELLE_VALEUR}
            if systeme == "CRM" and CRM_COL_SOCIETE: p["societe"] = r.ID_CODE_SOCIETE
            taches.append((r, sql, p, orig, "CRM" if systeme == "CRM" else "BPCUSTOMER", None))

            # ERP : restauration de BPDLVCUST, adresse par adresse
            if systeme == "ERP":
                if bk_dlv is None:
                    taches.append((r, None, None, None, "BPDLVCUST", None))   # pas de sauvegarde -> non restaurable
                    continue
                adrs = bk_dlv[(bk_dlv["ID_CODE_SOCIETE"] == r.ID_CODE_SOCIETE) &
                              (bk_dlv["ID_CODE_TIERS_ERP_1"] == r.ID_CODE_TIERS_ERP_1)]
                if "ALIGNE" in adrs.columns:          # seules les adresses réellement modifiées par l'update
                    adrs = adrs[adrs["ALIGNE"] == "True"]
                for a in adrs.itertuples():
                    v = None if pd.isna(a.VALEUR_ORIG) else a.VALEUR_ORIG
                    sql_d = _sql_erp_dlv(r.ID_CODE_SOCIETE) + " AND bpaadd_0 = :adr"
                    if not forcer: sql_d = garde(sql_d, col_rep_dlv(r.ID_CODE_SOCIETE))
                    p_d = {"val": v, "client": r.ID_CODE_TIERS_ERP_1, "adr": a.BPAADD, "attendu": r.NOUVELLE_VALEUR}
                    taches.append((r, sql_d, p_d, v, "BPDLVCUST", a.BPAADD))

        def _enreg(r, orig, statut, rc, table, adr):
            res.append({"RUN_ID": run_id, "SYSTEME": systeme, "CONNEXION": connexion, "TABLE": table,
                        "ID_CODE_SOCIETE": r.ID_CODE_SOCIETE, "ID_CODE_TIERS_ERP_1": r.ID_CODE_TIERS_ERP_1,
                        "ADRESSE_LIVRAISON": adr, "VALEUR_RESTAUREE": orig, "STATUT": statut, "ROWCOUNT": rc,
                        "HORODATAGE": datetime.now().isoformat(timespec="seconds")})
        if dry_run:
            for r, sql, _, orig, table, adr in taches:
                _enreg(r, orig, "DRY_RUN" if sql else "BPDLVCUST_SANS_BACKUP", None, table, adr)
        else:
            engine = crm_engine if systeme == "CRM" else erp_engines[connexion]
            with engine.begin() as conn:
                for r, sql, p, orig, table, adr in taches:
                    if sql is None:
                        _enreg(r, None, "BPDLVCUST_SANS_BACKUP", None, table, adr); continue
                    rc = conn.execute(text(sql), p).rowcount
                    _enreg(r, orig, "RESTAURE" if rc else "IGNORE_VALEUR_MODIFIEE", rc, table, adr)

    df_res = pd.DataFrame(res)
    df_res.to_csv(d / f"rollback_log_{datetime.now():%Y%m%d_%H%M%S}.csv", index=False, encoding="utf-8-sig")
    return df_res

# Simulation du rollback
#rollback(RUN_ID, crm_engine, erp_engines, dry_run=True)

In [14]:
# Rollback réel (à décommenter uniquement en cas de secours)
# rollback(RUN_ID, crm_engine, erp_engines, dry_run=False)